In [3]:
from vnstock import Vnstock

stock = Vnstock().stock(symbol="FPT", source='TCBS')
finance = stock.finance
income_statement = finance.income_statement(period='quarter').reset_index()

2025-06-27 12:06:30 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


In [15]:
from vnstock import Vnstock
def load_company_info(ticker: str):
    stock = Vnstock().stock(symbol=ticker, source='TCBS')
    company = stock.company

    overview = company.overview()
    # overview['website'] = overview['website'].removeprefix(
    #     'https://').removeprefix('http://')

    shareholders_df = company.shareholders()
    shareholders_df["share_own_percent"] = (
        shareholders_df["share_own_percent"] * 100).round(2)

    shareholders = shareholders_df.to_dict("records")
    events = company.events()
    events['price_change_ratio'] = (events['price_change_ratio']*100).round(2)
    events = events.to_dict("records")

    news = company.news()
    news = news[~news['title'].str.contains('insider', case=False, na=False)]
    news['price_change_ratio'] = (news['price_change_ratio']*100).round(2)
    news = news.to_dict("records")

    return overview, shareholders, events, news

In [16]:
overview, shareholders, processed_events, news = load_company_info('FPT')
overview

2025-06-27 14:19:48 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


,symbol,exchange,industry,company_type,no_shareholders,foreign_percent,outstanding_share,issue_share,established_year,no_employees,stock_rating,delta_in_week,delta_in_month,delta_in_year,short_name,website,industry_id,industry_id_v2
0,FPT,HOSE,Technology,CT,58429,0.406,1481.3,1481.3,2002,54646,2.8,-0.006,-0.021,-0.184,FPT Corp,https://www.fpt.com.vn,310,9537


In [20]:
from vnstock import Vnstock
company = Vnstock().stock(symbol='FPT', source='TCBS').company
df = company.profile()
print(df.loc[0, 'key_developments'])  # or any other long field

2025-06-27 12:50:04 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


 1.      Technology  Digital transformation based on technologies: AI, RPA, IoT, BigData, Cloud,…  Specialized solutions in fields: Banking - Finance, Public Finance, Telecommunication, Healthcare, Transport, Electricity, Water, Gas,…  Technological system integration and transformation.  Solution based on technology platforms: SAP, Oracle, Microsoft, ESRI.  Software testing and quality assurance services.  Microcircuit design, embedded software manufacturing, CAD/CAE…  2.      Telecommunication  Telecommunication services: Internet, exclusively leased line, data center, VoIP phone, value-added services, international and interprovincial connection, Cloud and IoT services… FPT television services: FPT television, FPT Play, Internet and mobile platform-based entertainment services. Digital content services: digital newspapers including VnExpress.net, Ngoisao.net, iOne.net, online advertisement, eClick AdNetwork smart advertising system.   3. Education  Elementary, secondary and highscho

In [8]:
df

,symbol,company_name,company_profile,history_dev,company_promise,business_risk,key_developments,business_strategies
0,VCB,Joint Stock Commercial Bank For Foreign Trade ...,Joint Stock Commercial Bank For Foreign Trade...,"October 30, 1962: Bank for Foreign Trade of ...",None,Risks related to geopolitical instability and...,Account service Mobilizing capital service; ...,"As the leading bank in Vietnam, one of the..."


In [1]:
import pandas as pd
import time
from tqdm import tqdm
from vnstock import Vnstock, Screener

def load_tickers(checkpoint_path="tickers_progress.csv",
                 checkpoint_every=10,
                 sleep_between=1.0):
    # 1) pull initial universe
    screener = Screener(source="TCBS")
    df0 = screener.stock(
        {"exchangeName": "HOSE,HNX", "marketCap": (2_000, 9_999_999_999_999)},
        limit=1700,
        lang="en"
    )
    tickers = df0["ticker"].tolist()

    # 2) try to resume from existing checkpoint, if present
    try:
        done_df = pd.read_csv(checkpoint_path)
        done_set = set(done_df["ticker"].tolist())
        results = done_df.to_dict(orient="records")
        start_idx = len(results)
        print(f"Resuming from {checkpoint_path}, {start_idx} tickers already done.")
    except FileNotFoundError:
        results = []
        done_set = set()
        start_idx = 0
        print("No checkpoint found—starting fresh.")

    # 3) iterate, fetch company info for each ticker
    for i, ticker in enumerate(tqdm(tickers[start_idx:], 
                                   desc="Fetching tickers", 
                                   unit="stk"), start=start_idx):
        if ticker in done_set:
            continue

        try:
            stock = Vnstock().stock(symbol=ticker, source="TCBS")
            company = stock.company

            # pull profile & overview
            prof = company.profile().iloc[0]
            over = company.overview().iloc[0]

            results.append({
                "ticker": ticker,
                "name": prof["company_name"],
                "short_name": over["short_name"]
            })
            done_set.add(ticker)

        except Exception as e:
            print(f"[{ticker}] ERROR: {e} — skipping")

        # checkpoint every N
        if (i + 1) % checkpoint_every == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)
            print(f"Checkpoint: saved {len(results)} rows to {checkpoint_path}")

        # simple rate-limit throttle
        time.sleep(sleep_between)

    # final save
    out_df = pd.DataFrame(results)
    out_df.to_csv(checkpoint_path, index=False)
    return out_df


In [2]:
load_tickers()

No checkpoint found—starting fresh.


Fetching tickers:   4%|▍         | 9/210 [00:20<07:45,  2.32s/stk]2025-06-27 14:33:13 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


Checkpoint: saved 10 rows to tickers_progress.csv


Fetching tickers:   9%|▉         | 19/210 [00:44<07:25,  2.33s/stk]2025-06-27 14:33:37 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


Checkpoint: saved 20 rows to tickers_progress.csv


...

Checkpoint: saved 200 rows to tickers_progress.csv


Fetching tickers: 100%|█████████▉| 209/210 [08:19<00:02,  2.39s/stk]2025-06-27 14:41:12 - vnstock.common.data.data_explorer - INFO - TCBS không cung cấp thông tin danh sách. Dữ liệu tự động trả về từ VCI.


Checkpoint: saved 210 rows to tickers_progress.csv


Fetching tickers: 100%|██████████| 210/210 [08:21<00:00,  2.39s/stk]


,ticker,name,short_name
0,AAA,An Phat Bioplastics Joint Stock Company,An Phat Bioplastics
1,ACB,Asia Commercial Joint Stock Bank,Asia Commercial Bank
2,ACG,An Cuong Wood - Working Joint Stock Company,An Cuong Wood
3,AGG,An Gia Real Estate Investment And Development ...,An Gia Real Estate
4,AGR,AgriBank Securities Corporation,AgriBank Securities
...,...,...,...
205,VRE,Vincom Retail Joint Stock Company,Vincom Retail
206,VSC,Viet Nam Container Shipping Joint Stock Company,Vietnam Container Shipping
207,VSH,Vinh Son - Song Hinh Hydropower Joint Stock Co...,Vinh Son - Song Hinh Hydropower
208,VTP,Viettel Post Joint Stock Corporation,Viettel Post
